# Face Mask Detection — Dataset Visualization

Visualize the class distribution and image counts in the `face-mask-detection-processed` dataset across **train**, **val**, and **test** splits.

In [ ]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = Path("/home/khoanguyen/workspace/UIT/face_mask_detection")
DATASET_DIR = BASE_DIR / "datasets" / "face-mask-detection-processed"

SPLITS      = ["train", "val", "test"]
CLASS_NAMES = {0: "With Mask", 1: "Without Mask", 2: "Mask Weared Incorrect"}
COLORS      = ["#2ecc71", "#e74c3c", "#f39c12"]   # green / red / orange

## 1 — Count images and class instances per split

In [ ]:
def count_split(split: str):
    """Return (num_images, Counter of class_id -> instance_count) for a split."""
    img_dir   = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    num_images = sum(
        1 for f in img_dir.iterdir() if f.suffix.lower() in image_exts
    ) if img_dir.exists() else 0

    class_counts = Counter()
    if label_dir.exists():
        for txt_file in label_dir.glob("*.txt"):
            for line in txt_file.read_text().splitlines():
                line = line.strip()
                if line:
                    cls_id = int(line.split()[0])
                    class_counts[cls_id] += 1

    return num_images, class_counts


split_stats = {}
for split in SPLITS:
    n_images, cls_counts = count_split(split)
    split_stats[split] = {"images": n_images, "counts": cls_counts}

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for split, info in split_stats.items():
    row = {"split": split, "images": info["images"]}
    total_instances = 0
    for cid, cname in CLASS_NAMES.items():
        cnt = info["counts"].get(cid, 0)
        row[cname] = cnt
        total_instances += cnt
    row["total_instances"] = total_instances
    rows.append(row)

# Add a totals row
totals = {"split": "TOTAL", "images": sum(r["images"] for r in rows)}
for cname in CLASS_NAMES.values():
    totals[cname] = sum(r[cname] for r in rows)
totals["total_instances"] = sum(r["total_instances"] for r in rows)
rows.append(totals)

df = pd.DataFrame(rows).set_index("split")
print(df.to_string())
df

## 2 — Image count per split (bar chart)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

split_labels = SPLITS
img_counts   = [split_stats[s]["images"] for s in SPLITS]
bars = ax.bar(split_labels, img_counts, color=["#3498db", "#9b59b6", "#1abc9c"], width=0.5)

for bar, val in zip(bars, img_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(val), ha="center", va="bottom", fontweight="bold")

ax.set_title("Number of Images per Split", fontsize=14, fontweight="bold")
ax.set_xlabel("Split")
ax.set_ylabel("Number of Images")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 3 — Overall class distribution (all splits combined)

In [ ]:
total_counts = Counter()
for info in split_stats.values():
    total_counts.update(info["counts"])

class_labels = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
class_values = [total_counts.get(i, 0) for i in sorted(CLASS_NAMES)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].bar(class_labels, class_values, color=COLORS, width=0.5)
for bar, val in zip(bars, class_values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 str(val), ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Total Instances per Class", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of Instances")
axes[0].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].tick_params(axis="x", labelrotation=10)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_values,
    labels=class_labels,
    colors=COLORS,
    autopct="%1.1f%%",
    startangle=140,
    pctdistance=0.82,
)
for at in autotexts:
    at.set_fontsize(11)
axes[1].set_title("Class Share (all splits)", fontsize=13, fontweight="bold")

plt.suptitle(f"Total instances: {sum(class_values):,}", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 4 — Class distribution per split (grouped bar chart)

In [ ]:
x         = np.arange(len(SPLITS))
bar_width  = 0.25
n_classes  = len(CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 5))

for i, (cid, cname) in enumerate(CLASS_NAMES.items()):
    values = [split_stats[s]["counts"].get(cid, 0) for s in SPLITS]
    offset = (i - n_classes / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, values, bar_width, label=cname, color=COLORS[i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(val), ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(SPLITS)
ax.set_title("Class Distribution per Split", fontsize=14, fontweight="bold")
ax.set_xlabel("Split")
ax.set_ylabel("Number of Instances")
ax.legend(title="Class")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 5 — Stacked bar: class share within each split

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bottoms = np.zeros(len(SPLITS))
for i, (cid, cname) in enumerate(CLASS_NAMES.items()):
    values = np.array([split_stats[s]["counts"].get(cid, 0) for s in SPLITS], dtype=float)
    bars = ax.bar(SPLITS, values, bottom=bottoms, label=cname, color=COLORS[i], width=0.5)
    for bar, val, bot in zip(bars, values, bottoms):
        if val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bot + val / 2,
                str(int(val)),
                ha="center", va="center", fontsize=9, fontweight="bold", color="white"
            )
    bottoms += values

ax.set_title("Stacked Class Distribution per Split", fontsize=14, fontweight="bold")
ax.set_xlabel("Split")
ax.set_ylabel("Number of Instances")
ax.legend(title="Class", loc="upper right")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 6 — Per-split pie charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, split in zip(axes, SPLITS):
    values = [split_stats[split]["counts"].get(cid, 0) for cid in sorted(CLASS_NAMES)]
    total  = sum(values)

    # Only draw wedges for classes that are present
    present_vals   = [v for v in values if v > 0]
    present_labels = [CLASS_NAMES[i] for i, v in enumerate(values) if v > 0]
    present_colors = [COLORS[i] for i, v in enumerate(values) if v > 0]

    wedges, texts, autotexts = ax.pie(
        present_vals,
        labels=present_labels,
        colors=present_colors,
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.82,
    )
    for at in autotexts:
        at.set_fontsize(10)
    ax.set_title(f"{split.capitalize()}  (n={total:,})", fontsize=12, fontweight="bold")

plt.suptitle("Class Distribution per Split", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()